© 2026 by Tamás Takács is licensed under CC BY-NC-SA 4.0. To view a copy of this license, visit https://creativecommons.org/licenses/by-nc-sa/4.0/

English translation managed by Tamás Takács. The translation was produced with AI assistance.

# Estimating the number of snowmen

In the Kingdom of Upper AIland, snowman building has a particularly long tradition. The snowmen they build do not differ much from the ones found here, but since it is a fictional territory, they put serious effort into keeping track of their snowmen. Thanks to this, they have accumulated an enormous amount of data from snowman-related statistics.

The ruler of Upper AIland asks you to use the statistics collected so far about the snowmen that are so important to them, and to produce an estimate of the number of snowmen based on the potentially related features.

Given that the kingdom is quite large, a significant number of snowmen can be found there throughout the whole winter.
Because of this, it is very hard to give an accurate estimate of the actual number of snowmen. To measure accuracy, instead of the mean squared error (MSE) you therefore have to minimize its square root (RMSE).

## Data

The related statistics are the following (taken for one week):
- `temperature` - Average weekly temperature (float, °C)
- `wind_speed` - Wind speed (float, m/s)
- `wind_direction` - Wind direction relative to north (float, radians)
- `precipation` - Average daily amount of precipitation (float, mm)
- `freeze_days` - Number of frosty days in the week (int, the number of days on which the average temperature was below freezing point)
- `carrot_price` - The price of carrots per kilogram (float, royal unified currency/kg)
- `scarf_tag` - Whether there was a sale on scarves in the clothing stores (bool, 0~no, 1~yes)
- `action_movies` - The number of action movies released that week (int, count)
- `log_consumption` - Firewood consumption of an average household (float, $m^3$)

**Target variable**
- `Hoember` (float) - The number of registered snowmen (only for the training data)

In addition, every record contains a unique identifier (`ID`), which is important for testing.

## Computing the error

A regression model gives the estimated output value based on the input features. Here is an example on a small dataset:

| Sample | True value | Predicted value |
|-------|--------------|-------------------------|
| 1     | 1242       | 1235.043                    |
| 2     | 0.0            | 30.65                    |
| 3     | 324            | 566.74        |
| 4     | 0.0            | 42.43     |
| 5     | 2133.64            | 1799.21   |
| 6     | 642            | 1026.7   |

---

The root mean squared error is computed with the following formula:

$$\text{RMSE}(\hat{y}, y) = \sqrt{\frac{\sum_i^n (\hat{y}_i - y_i)^2}{n}}$$

Let us look at the squared errors for the examples above:

| Sample | True value | Predicted value | Squared error |
|-------|--------------|-------------------------|----------|
| 1     | 1242       | 1235.043                    |   48.399849        |
| 2     | 0            | 30.65                    |   939.4225     |
| 3     | 324            | 566.74        |   58922.7076     |
| 4     | 0            | 42.43     | 1800.3049       |
| 5     | 2133            | 1799.21   |  111415.7641      |
| 6     | 642            | 1026.7   |  147225.69      |


Their average is: 

$$\frac{48.399849 + 939.4225 + 58922.7076 + 1800.3049 + 111415.7641 + 147225.69}{6} \\ \approx 53392.04815817$$

So the root mean squared error is:

$$\sqrt{53392.04815817} \approx 231.0672$$

Detailed description: [Link](https://en.wikipedia.org/wiki/Root_mean_square_deviation)

In what follows, we load the data, split it into training and test sets, and "train" a simple model on it.

You have to achieve an RMSE value **lower** than the result returned by this model.

## Importok

In [45]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.dummy import DummyRegressor

## Loading the data

In [46]:
labeled_df = pd.read_csv("train_labeled.csv")
labeled_features = labeled_df[labeled_df.columns[1:-1]].to_numpy()
labeled_out = labeled_df[labeled_df.columns[-1]].to_numpy()

test_df = pd.read_csv("test.csv")
test_features = test_df[test_df.columns[1:]].to_numpy()

print(labeled_features.shape, labeled_out.shape)
print(test_features.shape)

(1000, 9) (1000,)
(1000, 9)


In [47]:
labeled_df.describe()

,temperature,wind_speed,wind_direction,precipation,freeze_days,carrot_price,scarf_tag,action_movies,log_consumption,Hoember
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,-4.724559,9.926591,3.128444,17.572980,4.025000,199.756640,0.493000,2.399000,49.649827,59729.836000
std,14.560306,5.753930,1.816457,13.384211,1.974405,29.074566,0.500201,1.720686,23.594678,69204.329237
min,-29.768399,0.002694,0.000073,0.027385,1.000000,92.538122,0.000000,0.000000,10.108442,0.000000
25%,-17.938598,5.182556,1.538522,6.239041,2.000000,180.229321,0.000000,1.000000,29.258930,0.000000
50%,-4.165223,9.960682,3.144618,14.628117,4.000000,199.816577,0.000000,2.000000,49.218028,15309.000000
75%,7.912233,14.865868,4.714850,25.956475,6.000000,218.915476,1.000000,4.000000,70.730257,133807.250000
max,19.970686,19.966950,6.280406,65.319592,7.000000,290.441351,1.000000,5.000000,89.976611,212396.000000


## Splitting into train and eval datasets

In [48]:
train_x, eval_x, train_y, eval_y = train_test_split(
    labeled_features, labeled_out, test_size=0.2, random_state=43
)


## Training the model

In [49]:
model = DummyRegressor()
model.fit(train_x, train_y)

print(root_mean_squared_error(eval_y, model.predict(eval_x)))


66228.86166921744


## Evaluating and saving the test predictions

In [50]:
test_out = model.predict(test_features)

pd.DataFrame({"ID": test_df["ID"], "Hoember": test_out}).to_csv(
    "prediction.csv", index=False
)